# UEL_SGS_LucViHoangDe_C1

**Chosen approved backbone:** `distilbert/distilbert-base-multilingual-cased`

**Team:** Lục Vị Hoàng Đế — UEL / SGS.

Fine-tune one approved multilingual encoder; compare cross-entropy against square-root class weighting on a fixed group-aware validation split, then retrain the selected configuration on all training rows. Metric: macro F1 over the three exact competition labels.

Attach the competition and the approved [Kaggle model](https://www.kaggle.com/models/aisuko/distilbert-base-multilingual-cased/transformers/default) in Input, enable a GPU, then Run All. The notebook includes its Python source and computes split IDs locally and does not fetch GitHub or an external pretrained backbone. Dependencies should be prepared before an offline run. It creates a CSV but never submits to the leaderboard.

The source Overview/Data were supplied by the participant. Live Rules could not be independently verified because Kaggle returned rate limits; check Rules/Discussion before entering. No external data, translation model, pseudo-labeling, or multi-backbone ensemble is used.

In [ ]:
import os, sys, json, base64, zlib
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
MODEL_ID = 'distilbert/distilbert-base-multilingual-cased'
# Set an explicit attached-model directory only if discovery finds multiple inputs.
MODEL_DIR = None
DATA_DIR = None
RUN_WEIGHTED_COMPARISON = True
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
os.chdir(WORK)
import torch, transformers, numpy, pandas, sklearn
assert torch.cuda.is_available(), 'Enable GPU before a full training run.'
assert transformers.__version__.split('.')[0] == '4', 'Use transformers>=4.56,<5 as in requirements.txt.'
print({'torch': torch.__version__, 'transformers': transformers.__version__, 'GPU': torch.cuda.get_device_name(0)})

In [ ]:
# Embedded, reviewable source; no downloaded executable code.
payload = 'eNrtfWtz20ay6F/BUWpPgTYIUZLtTZRl6jqOnfXGcVy2k/1AsbAgORRhgQADgJK1uvrvt1/zAkBKck7q7Ie7D4vAzPT0zPT09HNwc7CpykbNm7I6TJKsyJokiTfXB6fBwRn9972CCovtPJvlKgrSDTxdqsVwls4vZmWhAnheb5pgnqd1nS2zedpkZREsyyp4//Prj8Hx6PhZzJDOirMiSS5VVUONJAnGwdnBKD6KR1h0EAUOJvM0z2YVgRJksvWmrJrgU10ClCX0GWzSZgW1Ail5B4/YgzwW2/XmOkjroNiYd5u0WMAb+N9mIUDqi1ylVRGvVVNl81oDWx4l9bysVLtWuVB5UqscsMRRSu2mSrMiaVTdJPUmz5ozjWI8L4u6SYvGAH7z/PuXb45f/6ArLNIm1WVXVdaohEeI/12oZaAnQoXq80ZV2VoVzeD0rAjgPzyrH7ewCIDLKs2XQbNSwSU0WfAqEHQYNAy5VnWN1bBGCf9U1CDGfhDWx1UGxRtslebQJM2va3iz3tYwlWUTzFQAY1lua7UIrrJmFaRBoa6CDJDcKPingFmA8Qtm85WaX2zKDN7ayUrzSqWL64BgIBpXqzL30K1VE7tD49924EAxuMruVHCVZZWuFZRuFjH2kczrS6dScAjwbDfJplKLjHCqY6h5diBQ8vI8g5UaM7jJZHl2QK+Smzydqfz27IDImh5g5GYtp9O4KRMiuFBAXWsoMdWO1+km1NUHndoNLGHEkwbt2tQUFps4rdLiXIW5KkICOxhEAdfI/q3Go/hpFECNRblOgNwaNT46+WvEsPf/p25wky2vxxrZ4nybnqvgMcxYAuN97I5CsEWy5O0SZovPUTDL0lqTJP6nUs22Ag6Ql2kT6o0UXk+g8jQKQp5legLw1BiGd75OP4dHMCrqqR5PRlFwFAXH0CIFjgE4jc8O1um8Ks+AU/xbVWWyyC4z5CTj0UAww9cwgTBf+KsOT+T9DGcKe4r45/IIamGVSA+EVwBfaYqChf6EixwOaXZH8Qj/eeoOFOtsuM4RFvs1Af945FYnTAAJxjCtqvQ6nFDtT1GwmQ78mkCvWyRpD0GaLb9etpSq35mxPQ6O1PDr0+76984DPxMMma20VgkRI3eoFokmTUFGCmm6otZL04fgyfsfdvzYFiHSHuTvxk6vgD/Myug4ADpQ1AlDqlS9zZEH3JwdaKgE7gzOB/0C9lae1U0IeJ0dzGFL4KZXpppBwdZrT9PZAcx1VpwnRG0wRwmilmeFQgBdeol6WpiOTZ/Lo56OmCsji+rvzF2J/tp+R86U9vU2n6sNTBFVLcuc+EpxHeqpG7hDqcormjDkODjYQQsDt5w67JvIOWzdtMFqZwfPTVs+qa6ADFYqXwTlFo5bPAuJMrh7ICN4i1WAV19m5bbO5dyA3dZ3uMTBCzidsmqNZykKKNgRnXcpVIKTaxEQLz2sFQCZKdi6Cggqv4bOsFa9TvMcICOXrePgbRnA4TE07LBZAfHBabUIllnTUJuK+K+wK8INpJxbngN7kLdPIVeuoYMemBnTtWyXTQXjCrEoXsARUYdcGtFJWzTjY1iIZb6tV+OP1VYN9M4gjstVO6JUnvkiFDDbTVrVyohFd8tUXVkl3S7gYMQ3kT7BVVJvZ+usRp4c9QozazjYQs0SCYcKtrPGJ35enW9xpt5RSbhQ9bzKSCQB3m+ESSuCqmIO8hjw4GwjW0bznO0MhQECE6eLBSLGTzVCbQDcvFyvYe/w/P++zUAkcGeUhgcwoCEB4NYhbAEsMB3RE1VIBXeoMhzitOyF3G0Cu2CzbbARTFQKywg4phUczem8qQ/9XklA6EOOCvxqD0NuRxMWavfhdkhVPGlqByyUIrPzByJAcvcQifOBDe209jVaZijs9kwkFYBkZQZDLx42lzuaWIbwwIZ3z8GOhvtJixrdNU4+Pg2LIvG5b96kqF31YRO3s5Fl/A9uup8Odjbzxj3vpxWjobkVv2TdNRvt60SXmT70i4dN7e5WlnU/vK2eXM3hqvPasl/6g/Vrre2A8IePsbDgYIx2AOFwjtBqT5iQqvNZQz+5QwEHrL8PoPBCV1ynQ4ze4wnu6e7O6pj67ZIOGsTzIt03sjV5YDMBbtUHIGx5zp04m6q2nrzp4GjR34+bPDiC+w40zRa/E0vNJ0CKq7ZG63b0RPuyg7bd6l+GpbMpO3g6MphG1VY3da3hpTWPpuNaeUqvKCc9whBDcIUjM1xHmdPj4JG2tDdHovSmw5Md7yc/Dlgeg+6SpADdHg1xOGVJguJZkpgZY2mtI05qi5YIlWTV+AADn5wdfP/y7esf3yIDOTv4x/PXb75///L5T/z47v0vP7/7mLx++4+XLz6+/gUqTc8KbRFBnY4E6dMgI606i6yVRRXAaWghuKvBLTZ8++Ovz398Kf1uGu5kWfHfTJ5VQd2cFV8FvxSgQTQrVaPBUhGpoiIgRswapl6hyL8G8R7fU11gwxvY4vBMcucvl6q6zNRVfFY8fwfj+e0lIa4NVgtQJ7N8pqrm0P4cogo3XMMKZCChgjKRD+eopLFOlGb19qK8R/VDQLioYWZgJupDOcZhjLrvT6vtcJ7Xm8P1+vuX7z8OSZ3x+livCT4VPBQaItUHDN/fBetVOlegbF48f334OV8PqxJbpl2Q7cIvhJvD7tgNmErvgnxelue5OqwUDdKDJe/ugiCbJc2gjDu+POGVZVv5MCs+sdo6vDz2KeHu+nd2njZVWWzrYd3AAVoeXpX5cgjV0EhbdQB6ve+velfH2Ww9PIcqwKgO5e/wJD4ZHs8ARt1U27k/mTvq3NUN8FCokcPk/sy/hn/9fvhaGg8vR/GJT6q6ku2BKu3u5hYZxsu3L3754eV75C+1akK93wfB0G74ew34QXhz7z6/xVPC199XaY06ekt9l4fKqvXbIkP9mOTBP8Nj0u8L+cAWZeCaP1bldvPTqzJf3OUP+RAFhqNba4Fz5rG4Us4+WeNBs9LeAPw9sK9R4oSTLl5fLLIq5IeaBNkoUJ9h7pPywhP+sQ331ajP3tkJHUKbot5WIMHW8ywbv0rh2LeHKZzkeV5ewTlacNEA68OsA/MGLWvbLIdfk1ysx1Sv0uOnzxhlGQqOQ5Y0lmLBjPwsdohxuVEFCNzVDEDikixbtuhZXs4v8MiEsVRhnq5ni/Q0WJJLJDwaHT8JHgX4B5CcoXulbZxexdsNii4hAfJtS6t4pT4vsnOFNlM7HvK2kIwOk00Eex2Rw4aMs+ic4KnXXZlaevHMC9eTg+rDza31/oj9+Maia+V7PP+zBZ/3uH78S1vu9BPIEiAKRB4AGMo92vuN6nS9yV3ZrgWh1dOtdSagqBUF8zLfroua5BoZWAyrta5Dz7HQ486yM3cYLM8ObhDeLRtcQLFvrjdqDLwkCi6U2iTCz4AuhSw9MZNdOoKKMYQH/zXW6LXoAmYaxKbf0EPwsqrKKjT9n9rl0SO7kR+3EQx7C/L4TX93t0Zf9LDKFrD3QOiaAx0uwgEZqAdobTXFMEr8f7aBUvV7SITM1R6A9gwW+ALBms6C1z/0Y5SCDHgtuymoT4O6FwNB4cGIKDhjr0F9U8AlW927VIgEw1SBKg46ZPFY8p12g+BvfFoZXvoQRLbFRVFeFYHts4sN0fZ+VNBNaPEgef2LkKCuBq0tUU+w3lT7VrXlD12hUcBbE8pCXXfKG8/dbNYE4TMpGgu5V5sKnZD4Q8kPBjygDRIePRmNRlHwzPzreT7bQzs7eKn3x9GTCGqL7RZ7e8bPwIUOBXX2qVgUcQ6B1LLFIPhvfsANcEeHH7GDQ/JNgGJVgs6Sp5sOVAKEA8In7v4eoB1cX/9QAzNfLlVlYMtJ4a+HPSoKkLXIRpHAqiTnKByEyHB1p18Fv6KDp0RFbentTKoLJ+q3AckcsJ6g+9bYH84lKGpZBVs5OydrLoKMW24R1L1DQPTsrH7MfDrAP46AFBvsYJxvX/30AssJuxgVsCUIMSHscdn17nkOeOGBQJNqB3NVVgs8xaDvJQgLICVw/1fU/86p8GfyhvCMP5WghxPASXaaBY+Dk+mAtWSkbA4MQO/5iMmVaqKgCqr+rcXUTCf3V/MuiaxrazyKvxn5oSUkwVlsFxYIbxggLpL0gp+z4u8gwgzffPh7YFySwTpt5qBxmxiT5+i4+ZytsczWQm08U5ewdPAuwGNVN/wWpSsoR0EROpG3qKqLRVJ3v63R7KQ+p/NGr0jwj3Q+T6tFHLxhD12hoDoIZyAaBXW2zkAXzIDz1iDLzlc9UScsOMIS0nHVCb8Y6EGRFFSWTZi5u+dqlQEODGOSTXGnZS0+aAvH+rd5NW35+G2VbNoJtMhcVICm0UYEEp6LjjQmPNMB9kg/Z2YUzhpb0WuTZlUd6RlFcp5MI/i/3q9Hz4IZLGMdfA6ewMKBzpB+G7zdrt9do3Sfkw+WzWyFUlCtKGGNgAEGP6Xn5zQ98ws4afRuhSWkyAgOZInf058PGM4SPjnWtmVEvTinOmjowhCRR49OjoHc4ScFwzx7ouUhgLWFSs+e6DCQVuPRgxpv5xeqcSVTMVThxvXtVHwWuuxATzEOcNfe905bqgxA7bq0qOeTC2oxwZ8tomFKAAQ/DdqEB8uKYg3o+SFVoFAVZIsWYkK7CQUb27hl8vTphlEArDLHgl5egPaEOnSbTdo6mro0QtzGqfBV8JsCqRd4FOqjqwrYcc2sAvkBmurQbT9X37pEzLwA3i+QcOGIWMfe/NIBzz0FfwvasTKgpTZZsXWswaig0d6wkTtAHDEqtcnsGvhhqFU4kCwv1PEsrGN2T4dWDQTSIgWKg7aeDGKtT5HmkDVNjlIX0RVpCDXxPY3ndBdlkq2Wdh8KP2E6OY2Ct2UBctIjjffjYGbeDoK/BAYE7Kxvnnzz7K8nR0dwvAGUMAUVeXzkgL5Q17T1Q9zrkfQ0wQeA/+SUXmPI0QAfMRKO52PAA6FSc0wdPRtMXRO7nAG1mFkGvj4LPWNTRKC9QKalUVl5e8bnAAYaREE4GLTAfXLm1AJoi6e0bZBQMRKP6ff/GiKdfGptMuckGTtEhfKabTIAlQ3LDOhuCJcD57uxPZN7Yrh2butdW9uCRjLTO/ETH4/+Bt8/6XqKYQCiX/JMT6YD3WUmwIitebtFjsi23OKcqNOBPXUSRwP+AcSzV/gYynkkmuQY1G6QmJOUJTr8OeOfdsT8vFbNqoShTq0QjP3ajvyjUr8ltR7B8wmtX8fUKUV0ap10sLv5rLf57O7mfqNFVYJWb6Sv0OBG7o6FHnl7pCIh0HJEDjwrFjpeVseCI/7KU1/DSkig7Rh8pC9uo+06nseTH+5tkWMBlfFF1tARXUFHE097Vda1SyzAuaFiOGmqSceuM53iMPreQ0F2DqeHStCw99nFJSETX7ITFQcDDyfdwDSehOYnUxBudFE4Ub9rFc/gWNKlU02xpsqAYyPNsyOvYXAx2otkOTDorAAZ1yGehGNlxHTEI3atRM4A7guMopV3QyRHOYaPwtEoMc1IU3qdMSZvbEQ8jiTqduxEGPX2AYJi3TYS6ijFG8deaJR8awIUdf/WM/UtMxRYyCBLIND0cKoNuC3bJSCIxQNj6aOoYOlU48z92aeOLVHCp3w0OMAQMWgqCSanWN1kXm5hG4UUTU4+9EHUsY8m2ppjAfDzPWGgsOKDUA8EQRiDiJRYu9IpbVQksiadhQ5SkRniTmiqOG9WCQh8Fch3quLF9SeMRD+HLMmqnxhDCK+2mFZIFrXGFdbBPWiOROyA3A2NGHuPjD9odbO7Xqt/b7sJDrznpXNmCPsaORvahOzaV35TQoIXFeuimqRHyNu1jaDtRh8+6H17a0VxtKYcaiPKISjGMIOg+YG8vlANetwLdH/Pg2dPhiRbbtfanBCgOSF0tUx0r5Bkj0aOYVNl58B4taqPDDX+BvSY78ZfG/HrWw4XFasDa/vAh649Q0Ns/HRi5UdWI5RNweXEejYYNEeRu3dQfsfqzwcUsgQbpInEDsMJNRfCs2ngCiJM6yYdhakGTc+4Hq4ChYxv0rs3psZeegMNw98Hp5KVIbDj37cpaDq5giLmXb8TljCHmL8Qf4P/px/w62jQjW522DPKBAkHt+rAZsJM59ZkBVsQzg6+Cn6gtCSO+IrI6oL/vqD5JMMSqS0clDEv19AF0BE5EPMSo/DwneK1CJBNS8Q2J6KxPl4Hm+0sz1D7id1O/vWvf2n0HEcfYbrf1TeQ1i6w35zUJTritqib9nlAMcbcRJOfbUejxV/ZyB0Z6yZtBzIV4/Z1TG1x8AJ3K0fhyqFJUwMLNssWgF4cOFZfau3YyowBLRIbGOdflTCypfKi2GOKlsHV2rGwa9QXXF8pmjQLbaKkJd7lAO2NTZJpv0doO9V0A8mNICFmzLYscaqlggppzSZj7ctvcuSPhiLT+1YyLLjjevwUTWPb5TJXIst6GViIiBubnFCuFHAO/IGqEc6f7iv2xsJI6zENfNM9v5wYmFN9sEiB9DDt2vIxA6NCYtX2fDbv5iq9cP09JpJNsuBYXJ5OQba2GWtcaSLnBes4jECr3DBNrmPYn1sthl2t8Y6CnjY2d8829DIeNF0Ac1v4dOF62rUY6rpVObKADSs3PNRTXIBbZ9aJ++90TJIHRaoYl4pR7fZ7VD4QRotS1eRIY4sWu4he/2B9QWtVnZP1icHSYyjRp5QcwQqNDj2EF2UBO7RMyoI84Gj08IRloSWGaw82Qv/GOaL8qb+9z2AkdLh2PF0ujFYKA2MwaSPihfBOYbUAW9bNQtSEW2HJd8Fx+98PzDjdUGxneUx2F8CQF/snwdtTZHBKL5UkH3U9ZdANLZul4L7wUaFRo22cGtXU83vuVs0lH2YH2d8ZNoCzsp31Bgz0hD7cQSRmWCZigHKMZ4olu/w6yBaR7/8V7RdwoD7R60obD3DavyupgtmT9/GeOugZQkYnlHXWkudTo1oWc+WiSd5w6JX1tKyGU1Gc4DG6AO/o/HVBq4/oc1iApYGgMyMyencxjB/XvL33YEGchhOPHbo1i1q71NNuMiZhjXyTj0GrdutVo1xMXm1irz3EZCjIFbvNTO4UuMmiqVV0N7gKSy7IzeSMoE5Q9kFttFlloP5UJcY2YtPR7T0y6eSOAD8a74/dM+DfmpCwqBZJqj2+gSOhyuBUtBcS3COUjqOqe4PqOBlbDkdOxeYs3zE6BzTBPCx9HYP40fWA1t6a7b0MeQDyVehkY+u8XxTPsTu2YjnNKLB/YPLBPV+EQ3GOqqYzYXER2xnnEWH2pSnlnkZI6yTLg131rtu+HhsMPW4o4L4e65ViKZtoWUTINhY+Ei2qIERa73bjMOhLee4CTazBSdB0a7cMOS3TC5butdA4mrERxekiBaFTh1+t0/qiI7djuJUJd+rQoXBGbNkfh6WFUm8U04n+PWV2VmjDBwGqt+uQM6P3kdoE60556vXvLyM7maL92n9n6NbWQPMmpobR8Pibb7AXDe1vwcloxFYH+DH8hku1OQANKFiO1mjb4giDnLgN/hwe9bSiOn6zY9sMfz520YAWXHz6BQu318zxH7J63jHCcC0LVp+RWbi3kLT5MUtnHa6cRJrV7uTie3W43YGxO5Q7iyJrYhPmCBizYvx4WGfqBUTsztzp5Ev1XLEyFbehf5Cg91qHvhj0PBq4704RlVP8CzzTXW/CXjlAJ7r5gsD5vD8if01R9vJQ1jZSn6wF5rHJ1vq2IZTh6YBR5mxPa1yIyBZFHDNa/w9F9mskymq+EpD0O942WV57mf9ovHsDerYCrQF/g2itWzgpFbr2821TvpDcSPz9M0bwvSqrDyBdK5CeX3gHKdf5iD70DPbWHrjY84syz1NA859Zs3qXLhZ0cQSesmiJSivY3Su12IIEiIH0yVVarbebPfKTzvnwJSgk3L4LEKypIbKaVyQy6D3uRGCILXGwyxqiwIh9JYaUXGYVSOKOC/7s4MWv3795/iH55y/vf/rw7vmLl8mLX96+ev0j7/DTJ6Nvnp3aJIT/YyjorKA/bDrkJdKJf5TZkS3IEEIGmAfmtun8wc/CrYkhAyCU00X93xa84raPFU5ik2a5ufdpU85XtW57oi+naeYrCqDRBV9LbTQgdUuPpMd0Pt8Cqiyu1Y3aGMBP9IEL4jndfAKrJkcEoqyGT8XsrbLzFQhrap5e2/JRPDqScqKvhBJLvfJnXvtNeaUqr1zfk6MWBqVj7QhPFxnetmLTYQFDvr0FqhHDMkksGW4nM9MS1rddJxhMki7c4dpowQTQqRu59Q0vglm2jmR8FWt6IPUW+LreKHcFdp8d0K1i+noOSqcE8q45xhcTLPVVHoG+ysMkZLbDzwkRSzYalRumG6Z1h4Ju78bNAaYtEAgAtW8HUEu0zAqapZhpM2K0LNXJixYtytsuCWLs2dE9ZlEnVnOousYWli5rsstOpD5OzdEzDsLH1TPbEF89PTpm8wiUuBQJmIzuxkRbKByYAMyF42U7cVJ8QlFy6Kfx8p4WbJgndwAy0zp0Up2Io5KLocebIMPdFk5uEBp4KKEaM/SWMfngUR5ZUroCHZWhw+m0oJYtHTD7rCUgqPwqyQkmz53BgnpjQdy2bXxOn+GjR5LybZRxhYGWIFZeN+iuDPGZI7JhOsZP9ExJQCyWho5bwYbKtkv46F6nBfDkpL9wvl2kbo0E7VOdWmgpdXhIKH+9OrhlVbGoEWRRxDNYsNU6rS58BiXSRK0Sz+sL3Z6XcCqu1rg8qAEDGy0SZBDaLov/5ZPqJTGJxTvy89WhCB+GnoifOaxM/K4Rh/GhPBEJRbpMjnYCpWBRfHOrQJgEkdioXVY2ZHu2MWy2wmy7OKcQQu7Q3YRDi0+Mc1sD5WSwChw5iFZ7EGXIq9+XuyWA78U3nE7naYFsga6w4iQK6VZiHD0eUqVXOvgRsXTimR1zHV2r5KGuPaaWseo3QOGzslYyHtRGMIMDTpTaeBwd/QFe+9oD4NPWCMUknGGmw3cyKT3Bkq01fAzyQLdSphmVe76MWyLJaf9tirla4iqHsiwUBnt4qI/fTk+UEwL/Tk6xId5/iA9D3XxI8Aan027zntjrHqgMp9X8K9TRMOqOokx49XVkNF8qROsIMngwefHmw/Rw8uHlu+nh3+rvhDji1tzDZnHpI+6Ax3XppRDxkxJrTNKmAdmGLU/1hZTdRTwdMujmpfVME6I8sbFUqCxqGX8irjr0QWZTtne2g3wNizBxrvAw8AUpIMceEUpOAQ6h0VBaLc8xlEGtDdcibbQHiAUwoSrTlj88J6WMTkFgix7b82QR8VnL9LYc5ueqwE1XVrTCyLN/1G9Amd5xpAiCVjO0SNiex31IyN+IJNWrsroAFW882nl16SYDYlHrsroeO8dYVifpJezSdAYixiCyYxibXzshzkmHBNotxjtUytCZxw3ygzIhnQedI+Vy/LWhZ6DmugT0+T6SgXPOU6RFwqaMkHZIFOSiQi/UZTY3Jh4qJPkx9O+mQl3QOaAoNZyngICj7E97L2xfUEqzjnuDO2zzUU6lGwc3F6fBJbD3kPHBHLUioTxwlL3oKObwcNiQCI3A6tRl3IcX6IGyW+y2dX+NxTcFDR/0xEZ6SsgHyb9j/K3dklyb9KSjZxhVgsu7cGvKVT9AAt1cdi8imZnSo0c82nY6jHMbsGxviVfmBWMUgPjnmy38K8Ykn/JBFOPIY8ASDiwHpJsrvy0SLTuG+r4o9zomjukQocW5r1je6Dt/SGUqMOVFD3pP6DWsjBvMA/oY+oUlRGsQU/h1vd9RCKLv+629vpmbgNTLYG+/xYAiuQ4ahmj91EameGD4t50SPSL7BjWDuswvVeg7XZ0qfM8mH+h6mMAhSP/ojvMVvH5bNq8wVd3ITeamPaNrpRJ5ZhRWPkaJrP2+rDbvSvZShwV8eXAEa3OzMtK24bz8GGoSx7Hu5HrsSoOqm61FokkrSmPJUHyDf8x905qjQaFncOOEJntBUuiSJ80AzWNtBfQIVz9Zwn62BzisaAVMGiNPobl3cDM8ngDpnJWjL+n5/l3RXMtVrcaU5FWgwuNs4UgGPbWyxTFbt8cBOaA7Fu6Bzo18iYSNSuLxkJWXOl2q5ppMCxyWB2CzGTXOrzFGL0/nfPNUGpxIG6wcO2jIjO23oe6dSR7H2B2UyX1YZzWHDboXd3zh5FuZeocNy3H00ZHXXy1hrh+6S2oPKS+8joTgloboMdSuFug259NR5yiIKKUh9zT2jD48I84et7FJFjB6cszktLk7esSRj2ElZ24QQN/AuofDrtF10HDHx+D7RteyYOnMEI7QJQfNLCvo0Z3kvmgByhhkLXR84jBtvntaQwTt+o5gled5TpYRFWjXCPFmPLHTCiUSfbQa9sdmKURXLnYweGK+RngSj4JHMqTBIHj0SI/dM2d5kA7H+me8VqmJwubgANsdM2kWB0N53SPVnBxr6W/s0TN+hWEt3Jmr05v4+SJd/5N3M0a4wTjQ9YUCb16NNRtzrdiRZ7Me+8Ojd9qSRjmUdbIBKZUsm0QnDRw0KsudyWOywdkTWD0mTX3CNEBAAhegdXsw082mVDmxxGODQ7/DkROaaYrI3ev3aBfTscrjNxPcWvqYBP7mTHa63sQ/Ajf6QK/N6Xu3BCoSIdBTRuloN+0wC7z2+FRceSIRkGuaAUrgqeZtlC9E7jhMuyHMnG+n+DkbeOafurIBzp2I12gJDUckou9AnEUH5Dw9CUTePfDODvJjQ71qbeY0uIvhMQKjnt6N6YbTl5gbW4OO18LpwWvGTM6xAjE2miPegYe3txGcZgBu4JqVyCm7yTx5kOyJLAdv0olc2yvE+iE0/UkYMEZ7rTwTovMJDMrIQGVbtACgUf5wxcje6sAx+kvH6VAWwQ1Tzi0I/W06uKVbayK3QLjELeuIqj6kHY4odaL7UaNkdmBTxCOfMZBNrSMuUE9u3I/hBhRslqA4gUlLqK2DJqvcTmWXEpvMy5qTBgr2xUWulZcQSHZK0XoAyPciq2dbmdCdjsgfBdkOtTzKuvSm3IRWfx7s18X/DC2+pbJzJpgefRjiMPGmDDJw7ub/lvV2C9t2yY8rJXewU8Q26ldOG8kip2N+lV4q8hjOywrVS2bacS/GQAV4fgF97EQkCrqH2tAd8eA/wHxhPkfUMl+IXaKvQW3FjwIUAeB4/EEnTrcEabqpMM7HN2zoWCctLow9nte1uCK7FNNTvUR/iwqx574hiI6N/QABvkOp3tgUgOMOuTkjnjbCCm7oz63eUje4Yv5tcfbUjukP9Q8M0BLAgNxScPgvwlY7d+ejQwD/0g4IkXSRLHg22t0hi3g83l0BfdS4Q/6ym/xJzA2EY+BDhwj7XBg8zm1BPxIr9vRc/2BWnqN25nm2IU6YYKJp0is64jeKeu6RkK+uJNQpBZwRFihX8IwPdmKKg9uLpR4RX9wx6HXIdPvDoEEfrR3uECNDMiY98B94VrirC4t2ZFbwLxj3SIt62jeFrWS3m7MDOQhPmdBRciASZ8kPDxF5oVoil1AHRewBuWJhhQar0CPmQyLSKHhKFVWebjDYnb9+47RxzjHgec4ZN7jtS8CjPVNecXBldwAaQcGqDx/8WtN9cYiCY/cc4vAoT3m2CnSPBHfa+YZWYrhonxHeQjGGeB8CI8BXX3DQZVf1tr20WU15NXEDT6fmC1r+6xaprcXE2F9ZvvMlH/naezmTA0m0unGf0kX3G4ldWAzggLkzln5Z0zclG6Fy7130AHYHicGSanRbo2CuhWlUrlHLQcT/LAja2YDoqwzTz9mR2LOfrfPynoAHO+/j39GAXpiY0cST5aOWEngv4PQZMVo3A6V3R7qBzr0ElOnP91nF77bL7rjKjo/I3YuQdw9FJ9LocfBzD6cuNrQ+bltn/4kYU2yuJfuxfxvKLcLtmOt9G1k7W+775UgnF5+Oy9Hefcm1/GCEu1RkzKPkdt+ZTWyC/lr3NFUqvfCMERM0JTTkuBVGzFcAdlmxYcLt9huAmZxvthgnVJIenZzPPDhuWNFn8dLa2iFajfDy6UePTqLg5G57xOjL9Fy0kgtzdwxERiaIRLIQZ/cc89DwAnPHnbQDq1P3WjszVMrTS+YpwA99l6B8FMd4/zpfk6G8+kWGKNF1E66BXh5cy71/JZPxoHRC7BwAA3t9U9K5vUl6d43gkSTTOlnUjYeecUlrAwdLBTo+VOcjYbyTvCKGxBfFl9vKGrl2u/UkWZBH5V/BI1YSd4RWBOmvy/f06LES7Th38nD+IA2v09IZtQgkPIIenyTzNH42UyP0afM1qXWfw9W2gX22KK8KnH/eWk5wpLS/T2Ck9LgL7sRdsSlf++0t2t2RZeSBcumglcLOALeYrGI8ps792jq74l4OcaLKXu+33Vz6M0J2T7kfENJNMUGGIHtZNd2aeoltCRBK17XeuXh65+7as1vttyz3yAt/yKl8f2evOxNf7nL9MiflH+4bRRaSFNuu2v/a6avd73B6YZeGvbGSI+1TejtF20ZS/g878/t9n71ezz3uwL3akBdFdi+3YEtrsl839r+/LGm/tPdOu5/YonjLTiC4aTG4N9Nzvq4s6WNOF5POV4P1LYnGENuX3ybGsu6nqqc6y2VDt/cIlf072/BFA9aqhpeGS4KvRpZZxsQm4AVu9j76TwXsl0XQCKieO/Rayfx9qW89MjffA+dJ256k7XzCvf8DaHtRaDFLC2zia1Q0SSA72HeO1NDT1nh8+M42dlEhENcb1NNOiEM+jW0cPfvkUTuoxIIyEqp9tesiJFtj/21IRrr98+VX59IUc8yarwzac9b9vmBXXN115rqt/DPX+c5e35mLAWYeq2h/YNlqBvdjG+Zo9hM1AMzEOm2njhNtf/cdJf0hWGiPF38pfWKU++n95fi2aG44nvv5URbB7ZuW8O3ooa4tQXBy1P1If7UyEXOSlf91ftQfls4dr6nvZeUVct9N+QbzRr7/9oocSyyS6O/9UdKlkUidDwGiH8JajOi74Xnuf7MD+UccvNdfI3B4nlhTgJOXxXmNV6vRxwPNl7HkeCW/CSbELbd5Tp9plUPdRMveQyDmS7x3iMPO9u1KxDstZp2tKt98ZARqJ+MzEbUGs54pusHX/R6cJxoJDJOnglAwfZbf21wAhu6kY0gNS3cn8sYKJ/j2a13Pl1uw6Ej30uOmhOInUuoF1NBNMTGQxOhYit1gGik90kVO5IkUPfNbScIcFUkJyvLUve6gPy6OLjGg+H2upS1BNH/yztEECKL5tlzPujJCavH/V/dPWd2n/wur+zlfJ/LdywSXaP/S7v1M55++kMe7F/Lrvetolvk/d5s2KKX/gXVsVLpur922yCgSrLnmGf/15Rsz13OQ/bc1v//w4wfzHuHw2zfb+W/Z38u0OP/BLi5s6E2eXlO4mFQ72x4p9XQe/IY/5rPg7yVeXKpGoOzAj6OjEb6fLQ0IPHqRhpImoy9ZCGYJYJG4fSYvjtAuhuO8/X+92MHh'
embedded = json.loads(zlib.decompress(base64.b64decode(payload)))
for relative, contents in embedded.items():
    path = WORK / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents, encoding='utf-8')
sys.path.insert(0, str(WORK))
print('Embedded files:', sorted(embedded))

In [ ]:
from protector.constants import APPROVED
from protector.data import read_data, sha256, write_json, audit_data
from protector.training import train_experiment, finalize
assert MODEL_ID == 'distilbert/distilbert-base-multilingual-cased', 'This notebook is configured for the explicitly selected DistilBERT backbone.'
input_root = Path('/kaggle/input')
if DATA_DIR is None:
    roots = [p.parent for p in input_root.rglob('train.csv') if (p.parent/'test.csv').exists() and (p.parent/'sample_submission.csv').exists()]
    assert len(roots) == 1, f'Set DATA_DIR explicitly; candidates: {roots}'
    DATA_DIR = roots[0]
DATA_DIR = Path(DATA_DIR)
train, test, sample = read_data(DATA_DIR)
audit_data(DATA_DIR, WORK/'artifacts/audit')
if MODEL_DIR is None:
    slug = APPROVED[MODEL_ID].split('/')[1]
    candidates = [p.parent for p in input_root.rglob('config.json') if slug in str(p.parent).lower()]
    assert len(candidates) == 1, f'Attach the approved Kaggle model or set MODEL_DIR explicitly. Candidates: {candidates}'
    MODEL_DIR = candidates[0]
MODEL_DIR = Path(MODEL_DIR)
assert (MODEL_DIR/'config.json').exists()
print('Data:', DATA_DIR, 'Model:', MODEL_DIR, 'Rows:', len(train), len(test))

In [ ]:
runs = []
configs = ['distilbert_ce'] + (['distilbert_weighted'] if RUN_WEIGHTED_COMPARISON else [])
for name in configs:
    output = WORK/'artifacts'/name
    if not (output/'metrics.json').exists():
        train_experiment(DATA_DIR, WORK/'artifacts/audit/split.csv', WORK/f'configs/{name}.json', MODEL_DIR, output)
    metric = json.loads((output/'metrics.json').read_text())
    runs.append((metric['macro_f1'], output))
    print(name, metric)
selected_score, selected_run = max(runs, key=lambda item: item[0])
print('Selected:', selected_run, 'validation macro F1:', selected_score)
print('Small differences require confirmation on another seed/split before making strong claims.')

In [ ]:
final_dir = WORK/'artifacts/final'
if not (final_dir/'submission_validation.json').exists():
    finalize(DATA_DIR, selected_run, MODEL_DIR, final_dir)
validation = json.loads((final_dir/'submission_validation.json').read_text())
print(json.dumps(validation, indent=2))
from IPython.display import display, FileLink
display(FileLink(str(final_dir/'submission.csv')))
print('No Kaggle submission was made. Review and submit manually when authorized.')

## Output interpretation

Validation metrics belong to selection models trained on the training partition. The final checkpoint is trained again on all 14,000 rows for the selected number of epochs; its generalization score is unknown until Kaggle evaluates it. A valid CSV is not proof of a high hidden-test score. Class order is fixed as BENIGN, JAILBREAK, PROMPT_INJECTION. Original prompts are treated solely as untrusted training data.